# 01 — Baseline RAG pipeline (Phase B)

**Project:** SD-A-01-RAG — The Footprint of Retrieval-Augmented Generation Components  
**Phase:** B (Pipeline — end-to-end RAG). See `MASTER_PLAN.md` §8.

This notebook is **orchestration only**. All reusable logic lives in `src/`.

## Default baseline configuration (§8)

| Component | Setting |
|---|---|
| Chunk size | 512 tokens |
| Chunk overlap | 10% (51 tokens) |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` (384-dim) |
| Vector store | FAISS `IndexFlatIP` over L2-normalized vectors → cosine similarity |
| Top-k | 3 |
| LLM | `llama3.1:8b-instruct-q4_K_M` via Ollama, `temperature=0`, `seed=42` |
| Random seed | 42 (Python `random`, NumPy) |

## Sub-steps

- **B.1** — PDF → text extraction (`src/pdf_extraction.py`) ✅
- **B.2** — Chunking (`src/chunking.py`) ✅
- **B.3** — Embedding (`src/embedding.py`) ✅
- **B.4** — FAISS indexing (`src/indexing.py`) ✅
- **B.5** — Retrieval (`src/retrieval.py`) ✅
- **B.6** — Generation (`src/generation.py`) ✅
- **B.7** — Pipeline composer (`src/pipeline.py`) ✅
- **B.8** — Run all 13 evaluation questions, save answers, spot-check ← *this commit*

**Phase B does not include measurement plumbing** (CodeCarbon, peak RAM, token counts) — that is Phase C.

## Setup

Imports, deterministic seeds, and `sys.path` shim so the notebook can import from `../src/`.

In [1]:
import random
import sys
from pathlib import Path

import numpy as np

# Project root = parent of notebooks/. Ensures imports work regardless of
# where Jupyter is launched.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Determinism (§13.5). Torch seed is set later in B.3 once it is loaded.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PDF_DIR = PROJECT_ROOT / "dataset" / "pdfs"
TEXT_DIR = PROJECT_ROOT / "corpus_text"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PDF_DIR:      {PDF_DIR}  (exists={PDF_DIR.exists()})")
print(f"TEXT_DIR:     {TEXT_DIR}  (exists={TEXT_DIR.exists()})")
print(f"SEED:         {SEED}")

PROJECT_ROOT: /Users/ismaildemircan/Desktop/SD-A-01-RAG
PDF_DIR:      /Users/ismaildemircan/Desktop/SD-A-01-RAG/dataset/pdfs  (exists=True)
TEXT_DIR:     /Users/ismaildemircan/Desktop/SD-A-01-RAG/corpus_text  (exists=True)
SEED:         42


## B.1 — PDF → text extraction

Extract each of the 5 corpus PDFs with `pypdf`, apply minimal cleaning, write each as a `.txt` file under `corpus_text/`, and verify word counts against `dataset/corpus_stats.md` within ±5% (§8 B.1).

Cleaning is deliberately conservative — sentence boundaries are preserved so they survive into B.2 chunking. See `src/pdf_extraction.py` module docstring for the full policy.

In [2]:
from src.pdf_extraction import (
    extract_corpus,
    verify_word_counts,
    format_comparison_table,
)

stats = extract_corpus(pdf_dir=PDF_DIR, out_dir=TEXT_DIR)

# Show output paths and sizes.
print("Extracted .txt files:")
for short_name, info in stats.items():
    rel = info["txt_path"].relative_to(PROJECT_ROOT)
    print(
        f"  {short_name:>15s}  →  {str(rel):<35s}  "
        f"({info['words']:>9,d} words, {info['chars']:>9,d} chars)"
    )

# Sanity check against dataset/corpus_stats.md (raises if any doc > ±5%).
df = verify_word_counts(stats, tolerance=0.05)
print("\nWord-count comparison vs. dataset/corpus_stats.md (±5% tolerance):\n")
print(format_comparison_table(df))

Extracted .txt files:
         01_BLOOM  →  corpus_text/01_BLOOM.txt             (    6,939 words,    44,622 chars)
           02_IEA  →  corpus_text/02_IEA.txt               (  115,129 words,   775,888 chars)
        03_Google  →  corpus_text/03_Google.txt            (   56,660 words,   379,163 chars)
          04_EPRI  →  corpus_text/04_EPRI.txt              (   14,264 words,   100,126 chars)
    05_Greenpeace  →  corpus_text/05_Greenpeace.txt        (    7,025 words,    51,446 chars)

Word-count comparison vs. dataset/corpus_stats.md (±5% tolerance):

          doc  expected    actual    diff diff_pct within_tolerance
     01_BLOOM     6,939     6,939      +0   +0.00%               OK
       02_IEA   115,129   115,129      +0   +0.00%               OK
    03_Google    56,660    56,660      +0   +0.00%               OK
      04_EPRI    14,264    14,264      +0   +0.00%               OK
05_Greenpeace     7,025     7,025      +0   +0.00%               OK
        TOTAL   200,017   200,0

## B.2 — Token-aware chunking (full corpus)

Chunk all 5 documents at the §8 baseline configuration (`chunk_size=512`, `overlap_pct=10%`). Splitter: LangChain `RecursiveCharacterTextSplitter.from_tiktoken_encoder` with `cl100k_base`. See `src/chunking.py` module docstring for the tokenizer-proxy choice.

Output: chunk metadata persisted to `indices/<config_hash>/chunks.parquet`. The matching `faiss.index` for the full corpus is written in B.8 (full-corpus embedding step). The smoke test below builds a separate, in-memory FAISS index over `01_BLOOM` only to validate the wiring before that runtime cost is incurred.

In [3]:
from src.chunking import (
    DEFAULT_CHUNK_SIZE,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_OVERLAP_PCT,
    chunk_corpus,
    chunk_summary,
    config_hash,
    save_chunks,
)

CHUNK_SIZE = DEFAULT_CHUNK_SIZE
OVERLAP_PCT = DEFAULT_OVERLAP_PCT
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
HASH = config_hash(CHUNK_SIZE, OVERLAP_PCT, EMBEDDING_MODEL)
INDEX_DIR = PROJECT_ROOT / "indices" / HASH
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Baseline config:")
print(f"  chunk_size:      {CHUNK_SIZE}")
print(f"  overlap_pct:     {OVERLAP_PCT}%")
print(f"  embedding_model: {EMBEDDING_MODEL}")
print(f"  config_hash:     {HASH}")
print(f"  index dir:       {INDEX_DIR.relative_to(PROJECT_ROOT)}/")

# Chunk all 5 docs.
all_chunks = chunk_corpus(TEXT_DIR, CHUNK_SIZE, OVERLAP_PCT)
print(f"\nTotal chunks: {len(all_chunks)}")

# Persist chunks.parquet under indices/<hash>/. faiss.index follows in B.8.
save_chunks(all_chunks, INDEX_DIR / "chunks.parquet")
print(f"Saved → {(INDEX_DIR / 'chunks.parquet').relative_to(PROJECT_ROOT)}")

# Per-doc summary.
summary = chunk_summary(all_chunks)
print("\nPer-doc summary:\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))

# Sample chunk (first BLOOM chunk) for visual sanity.
sample = next(c for c in all_chunks if c.source_doc == "01_BLOOM")
print("\nExample chunk (first BLOOM chunk):")
print(f"  chunk_id:    {sample.chunk_id}")
print(f"  start_char:  {sample.start_char}")
print(f"  end_char:    {sample.end_char}")
print(f"  token_count: {sample.token_count}")
print(f"  char span:   {sample.end_char - sample.start_char}")
print(f"  text[:240]:  {sample.text[:240]!r}")

Baseline config:
  chunk_size:      512
  overlap_pct:     10%
  embedding_model: sentence-transformers/all-MiniLM-L6-v2
  config_hash:     6fb92835
  index dir:       indices/6fb92835/

Total chunks: 754
Saved → indices/6fb92835/chunks.parquet

Per-doc summary:

   source_doc  n_chunks  mean_tokens  min_tokens  max_tokens  p95_tokens  mean_chars
     01_BLOOM        24        489.6         401         506       505.0     2,015.9
       02_IEA       457        407.6           1         510       504.0     1,824.1
    03_Google       188        467.3          54         509       502.6     2,175.4
      04_EPRI        57        486.4         454         506       499.2     1,918.1
05_Greenpeace        28        458.1         220         505       501.6     1,976.0
        TOTAL       754        432.9           1         510       503.4     1,930.5

Example chunk (first BLOOM chunk):
  chunk_id:    01_BLOOM_0000
  start_char:  0
  end_char:    2070
  token_count: 490
  char span:   2070


## End-of-batch smoke test — B.3 → B.7 wired end-to-end

Restrict to `01_BLOOM` only (smallest doc, ~20 chunks). Validates that all six modules wire together correctly without burning the full-corpus runtime budget on a 13-question evaluation. The full-corpus embedding + index build + 13-question run happens in B.8 (separate batch, behind the >5-min plan-and-confirm gate).

Pipeline path under test: `chunks (BLOOM only) → embed (MiniLM-L6) → FAISS IndexFlatIP → top-3 retrieval → Llama 3.1 8B Q4 generation`.

**Note:** The first run of `embed_chunks` will trigger sentence-transformers' download of `all-MiniLM-L6-v2` (~90 MB) on first use. Subsequent runs hit the local HF cache.

In [4]:
import time

from src.embedding import embed_chunks
from src.indexing import build_index
from src.pipeline import RAGConfig, run_rag

# Filter the in-memory chunk list to BLOOM only.
bloom_chunks = [c for c in all_chunks if c.source_doc == "01_BLOOM"]
print(f"Smoke-test scope: 01_BLOOM only ({len(bloom_chunks)} chunks)")

# B.3 — embed.
t0 = time.perf_counter()
embeddings = embed_chunks(bloom_chunks, show_progress=False)
embed_s = time.perf_counter() - t0
print(
    f"\n[B.3] Embedding shape: {embeddings.shape}  "
    f"dtype={embeddings.dtype}  ({embed_s:.2f}s)"
)

# Verify L2-normalization — norms should all be 1.0 (within float32 noise).
norms = np.linalg.norm(embeddings, axis=1)
print(
    f"      L2 norms — min: {norms.min():.6f}  "
    f"max: {norms.max():.6f}  (expect ~1.0)"
)

# B.4 — build a tiny in-memory FAISS index.
t0 = time.perf_counter()
index = build_index(embeddings)
build_s = time.perf_counter() - t0
print(
    f"\n[B.4] Index ntotal: {index.ntotal}  dim={index.d}  ({build_s:.4f}s)"
)

# B.5–B.7 — run one query end-to-end.
SMOKE_QUERY = (
    "What is the total carbon footprint of training the BLOOM model?"
)
config = RAGConfig()
print(f"\n[B.5-B.7] Smoke query: {SMOKE_QUERY!r}")

result = run_rag(SMOKE_QUERY, config, index, bloom_chunks)

print(f"\nTop-{config.top_k} retrieved chunks:")
for r in result["retrieved_chunks"]:
    snippet = r.chunk.text[:140].replace("\n", " ")
    print(f"  [{r.rank}] {r.chunk.chunk_id}  score={r.score:.4f}")
    print(f"        {snippet}...")

print(f"\nTimings (s):")
for k, v in result["timings"].items():
    print(f"  {k:>9s}: {v:.3f}")

print(f"\nAnswer:")
print(result["answer"])

Smoke-test scope: 01_BLOOM only (24 chunks)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[B.3] Embedding shape: (24, 384)  dtype=float32  (3.30s)
      L2 norms — min: 1.000000  max: 1.000000  (expect ~1.0)

[B.4] Index ntotal: 24  dim=384  (0.0001s)

[B.5-B.7] Smoke query: 'What is the total carbon footprint of training the BLOOM model?'

Top-3 retrieved chunks:
  [0] 01_BLOOM_0015  score=0.7713
        Section 4, these numbers represent a small part of the actual carbon emissions and environmental impacts of training ML models, given that t...
  [1] 01_BLOOM_0016  score=0.7663
        Other Models 64,257 3.66 5.53% Miscellaneous Processes 57,961 3.30 4.98% 6B Model 51,686 2.95 4.45% Model Evaluation 43,172 2.46 3.71% Total...
  [2] 01_BLOOM_0013  score=0.7467
        recent similar LLMs, attempt to estimate the dynamic consumption of all processes run within the scope of the BigScience workshop and discus...

Timings (s):
      embed: 0.039
   retrieve: 0.001
   generate: 23.985

Answer:
According to Table 5 in chunk 2, the dynamic energy consumption and CO2 emissions o

## B.8 — Full-corpus baseline evaluation

Run all 13 evaluation questions through the §8 default-config pipeline. **This step crosses the >5-min plan-and-confirm threshold** (estimated 3.5–4.5 min on M4): ~30–60 s embedding 754 chunks, <1 s FAISS build/save, ~170–220 s for 13 Llama 3.1 generations.

Output: `results/baseline_answers.csv` with columns:

```
question_id, source_doc, question_type, question, answer,
retrieved_chunk_ids, retrieved_scores,
embed_s, retrieve_s, generate_s, total_s
```

List-typed columns (`retrieved_chunk_ids`, `retrieved_scores`) use `;` as a separator. Multi-doc `source_doc` uses `+` (only Q13). The full FAISS index is also persisted to `indices/<config_hash>/faiss.index` here so subsequent phases can reload (index, chunks) via `load_index`.

Spot checks rendered after the run: **Q07** (factoid, Google), **Q10** (numerical, EPRI), **Q06** (synthesis, IEA) — different docs from prior smoke tests so the eyeball is informative.

In [5]:
import time

import pandas as pd
from tqdm.auto import tqdm

from src.chunking import load_chunks
from src.embedding import embed_chunks
from src.eval_questions import load_eval_questions
from src.indexing import build_index, save_index
from src.pipeline import RAGConfig, run_rag

# Determinism (§13.5 / §C.5). Torch is loaded transitively by
# sentence-transformers; seed it now that it's importable.
try:
    import torch
    torch.manual_seed(SEED)
    print(f"torch.manual_seed({SEED}) set")
except ImportError:
    print("torch not importable — skipping torch seed (still deterministic via temp=0/seed=42)")

# 1. Reload chunks from parquet (validates persistence round-trip).
chunks_path = INDEX_DIR / "chunks.parquet"
all_chunks = load_chunks(chunks_path)
print(f"\nLoaded {len(all_chunks)} chunks from {chunks_path.relative_to(PROJECT_ROOT)}")

# 2. Embed full corpus.
print(f"\nEmbedding {len(all_chunks)} chunks with {EMBEDDING_MODEL}...")
t0 = time.perf_counter()
embeddings = embed_chunks(all_chunks, model_name=EMBEDDING_MODEL, show_progress=True)
embed_total_s = time.perf_counter() - t0
print(f"  shape: {embeddings.shape}  dtype={embeddings.dtype}  ({embed_total_s:.1f}s)")

# 3. Build + save FAISS index.
t0 = time.perf_counter()
index = build_index(embeddings)
save_index(index, all_chunks, INDEX_DIR)
build_save_s = time.perf_counter() - t0
faiss_path = INDEX_DIR / "faiss.index"
print(f"FAISS index: ntotal={index.ntotal} dim={index.d}  saved → {faiss_path.relative_to(PROJECT_ROOT)} ({build_save_s:.3f}s)")

# 4. Load eval questions.
eval_path = PROJECT_ROOT / "dataset" / "eval_questions.md"
questions = load_eval_questions(eval_path)
print(f"\nLoaded {len(questions)} evaluation questions from {eval_path.relative_to(PROJECT_ROOT)}")
qtype_counts = pd.Series([q.question_type for q in questions]).value_counts().to_dict()
print(f"  by type: {qtype_counts}")

# 5. Run all 13 questions.
config = RAGConfig()
rows = []
print(f"\nRunning {len(questions)} questions at config_hash={config.hash}, top_k={config.top_k}, llm={config.llm_model}")
run_t0 = time.perf_counter()
for q in tqdm(questions, desc="Eval", unit="q"):
    result = run_rag(q.question, config, index, all_chunks)
    retrieved = result["retrieved_chunks"]
    timings = result["timings"]
    rows.append({
        "question_id": q.question_id,
        "source_doc": q.source_doc,
        "question_type": q.question_type,
        "question": q.question,
        "answer": result["answer"],
        "retrieved_chunk_ids": ";".join(r.chunk.chunk_id for r in retrieved),
        "retrieved_scores": ";".join(f"{r.score:.4f}" for r in retrieved),
        "embed_s": timings["embed"],
        "retrieve_s": timings["retrieve"],
        "generate_s": timings["generate"],
        "total_s": timings["embed"] + timings["retrieve"] + timings["generate"],
    })
run_total_s = time.perf_counter() - run_t0

# 6. Save CSV.
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(parents=True, exist_ok=True)
csv_path = results_dir / "baseline_answers.csv"
df_results = pd.DataFrame(rows)
df_results.to_csv(csv_path, index=False)
print(f"\nSaved {len(df_results)} rows → {csv_path.relative_to(PROJECT_ROOT)}")

# 7. Summary stats.
non_empty = df_results["answer"].str.len() > 0
refusal_marker = "I don't have enough information"
refusals = df_results["answer"].str.contains(refusal_marker, case=False, na=False)
print(f"\nSummary:")
print(f"  total wall-clock for 13 LLM calls: {run_total_s:.1f}s ({run_total_s/60:.2f} min)")
print(f"  mean per-question total_s:         {df_results['total_s'].mean():.2f}s")
print(f"  mean generate_s:                   {df_results['generate_s'].mean():.2f}s")
print(f"  non-empty answers:                 {int(non_empty.sum())}/{len(df_results)}")
print(f"  refusals ('not enough info'):      {int(refusals.sum())}/{len(df_results)}")

# 8. Spot-check view: Q07 (factoid/Google), Q10 (numerical/EPRI), Q06 (synthesis/IEA).
SPOT_CHECK_IDS = ["Q07", "Q10", "Q06"]
q_by_id = {q.question_id: q for q in questions}
row_by_id = {r["question_id"]: r for r in rows}

print("\n" + "=" * 78)
print("SPOT CHECK — eyeball topical relevance against reference answer")
print("=" * 78)
for qid in SPOT_CHECK_IDS:
    q = q_by_id[qid]
    r = row_by_id[qid]
    print(f"\n--- {qid} [{q.question_type} | {q.source_doc}] ----------------------------------")
    print(f"Q: {q.question}")
    print(f"\nRetrieved chunks (top-{config.top_k}):")
    for cid, score in zip(r["retrieved_chunk_ids"].split(";"), r["retrieved_scores"].split(";")):
        print(f"  - {cid}  score={score}")
    print(f"\nGenerated answer ({r['generate_s']:.1f}s):")
    print(f"  {r['answer']}")
    print(f"\nReference answer:")
    print(f"  {q.reference_answer}")
    print(f"\nSource location: {q.source_location}")

torch.manual_seed(42) set

Loaded 754 chunks from indices/6fb92835/chunks.parquet

Embedding 754 chunks with sentence-transformers/all-MiniLM-L6-v2...


Batches:   0%|          | 0/24 [00:00<?, ?it/s]

  shape: (754, 384)  dtype=float32  (5.0s)
FAISS index: ntotal=754 dim=384  saved → indices/6fb92835/faiss.index (0.008s)

Loaded 13 evaluation questions from dataset/eval_questions.md
  by type: {'factoid': 6, 'synthesis': 3, 'numerical': 3, 'multi-doc': 1}

Running 13 questions at config_hash=6fb92835, top_k=3, llm=llama3.1:8b-instruct-q4_K_M


Eval:   0%|          | 0/13 [00:00<?, ?q/s]


Saved 13 rows → results/baseline_answers.csv

Summary:
  total wall-clock for 13 LLM calls: 201.7s (3.36 min)
  mean per-question total_s:         15.51s
  mean generate_s:                   15.32s
  non-empty answers:                 13/13
  refusals ('not enough info'):      4/13

SPOT CHECK — eyeball topical relevance against reference answer

--- Q07 [factoid | 03_Google] ----------------------------------
Q: In 2024, Google reduced its data centre energy emissions despite significantly growing its electricity consumption. What were the specific percentage changes in each metric, and what is cited as the primary reason this was possible?

Retrieved chunks (top-3):
  - 03_Google_0184  score=0.7122
  - 02_IEA_0454  score=0.6965
  - 03_Google_0075  score=0.6908

Generated answer (15.9s):
  I don't have enough information to answer that.

However, I can provide some context from the provided chunks:

* Chunk 3 mentions that Google achieved at least 80% CFE (carbon-free energy) in nine